# P9｜PAFA Projector Only

**Pipeline ID：P9**  
**研究问题：** 在 P2 的 BEATs AudioSet-only/native-head reference 上，只对真实 patient-ID eligibility 已验收的 lanes 加入 PAFA projector、且不加入 PCSL/GPAL loss，eligible-lane native task 表现是否改变？  
**Role：** ADP-PAFA-PROJECTOR architecture/capacity control。  
**状态：Design / Not Ready。**

## 合同、组件与唯一变化

**Verified Contract：** P9 的 PAFA projector 只作用于真实 patient-ID eligibility receipt 已通过的 lanes/rows；没有真实 patient ID 的 lanes 必须 bypass projector，完整保留 P2 native representation/head/loss 路径。P9 的 PCSL/GPAL loss weight 在所有 lanes 固定为 0。P9 只隔离 projector，不能称官方 PAFA reproduction；历史 ICBHI-test-selected R0 也不得当作 clean source encoder。

**Proposed Method：** 固定 P2 的 BEATs AudioSet-only encoder、native heads、source-proportional sampler、native CE/BCE、预算/seed/selection。对 eligibility matrix 标记为 `real_patient_id_eligible=true` 的 rows 路由到 source-defined PAFA projector；其他 rows 使用 identity/native bypass。Comparator 为 **P9 − P2，仅 eligible lanes 的 projector/匹配容量改变**。P9−P2 的方法效应和 go/no-go vote 只能归因于 eligible lanes；ineligible lanes 仅作为 native parity/guardrail，不贡献 projector effect。

**HOLD：** per-lane patient-ID eligibility、projector 维度/normalization、frozen encoder 与 projector/head trainable scope、parameter-matched control、checkpoint source/SHA、预算、seed、selection 与 go/no-go 待冻结；full encoder 与 server HOLD。

| Dataset | Unit / native head | Projector eligibility 与路径 |
|---|---|---|
| ICBHI | cycle / flat4 `[B,4]` | 真实 patient ID receipt 通过后 eligible；official split overlap 仍披露 |
| SPRSound | event / `[B,2]` + `[B,7]` | 真实 patient ID/group provenance receipt 通过后 eligible；inter terminal label join |
| HF | 15-s recording / observed-positive heads | **ineligible：native bypass**；date proxy 不是 patient；gap/unknown omitted |
| KAUH | recording / raw9 `[B,9]` | verified P-number patient ID eligible；B/D/E 同组；shared/diagnosis HOLD |

**Execution gates：** P2 freeze receipt、不可变 per-lane eligibility matrix、projector source/revision/hash、parameter parity、matched compute 与 `projector_scope_receipt` 必须全部通过。该 receipt 至少逐 lane 记录 eligible/bypass rows、unique patients、batches、projector calls、native-only calls 与 patient-ID provenance；ineligible lane projector calls 必须为 0。另需 local smoke、独立 verifier 与单独执行授权，否则 fail closed。

In [ ]:
import os
from pathlib import Path
PIPELINE_ID = "P9"
NOTEBOOK = Path("reproduce/P9_pafa_projector_only.ipynb")
STATUS = "Design / Not Ready"
PROJECT_ROOT = Path.cwd() if Path.cwd().name != "reproduce" else Path.cwd().parent
DATASET_ROOT = Path(os.environ.get("ACOUSTIC_DATA_ROOT", "dataset/raw"))
CONFIG = Path("experiments/P9_pafa_projector_only.yaml")
APPROVAL = Path("result/approvals/P9_execution_authorization.json")
PROJECTOR_SCOPE_RECEIPT = Path("result/receipts/P9_projector_scope_receipt.json")
PROJECTOR_SCOPE = "real_patient_id_eligible_lanes_only"
INELIGIBLE_BEHAVIOR = "bypass_projector_and_preserve_P2_native_path"
EFFECT_ATTRIBUTION_SCOPE = "eligible_lanes_only"
EXECUTION_ALLOWED = False
required = {"config":PROJECT_ROOT/CONFIG, "approval":PROJECT_ROOT/APPROVAL, "projector_scope_receipt":PROJECT_ROOT/PROJECTOR_SCOPE_RECEIPT}
dry_run_plan = {"pipeline_id": PIPELINE_ID, "projector": "PAFA source-defined; dimensions TBD", "projector_scope": PROJECTOR_SCOPE, "ineligible_behavior": INELIGIBLE_BEHAVIOR, "effect_attribution_scope": EFFECT_ATTRIBUTION_SCOPE, "patient_aware_loss_weight": 0.0, "per_lane_counts_required": ["eligible_rows","bypass_rows","unique_patients","projector_calls","native_only_calls"], "missing_gates": [n for n,p in required.items() if not p.is_file()]}
assert NOTEBOOK.name.startswith(f"{PIPELINE_ID}_") and not EXECUTION_ALLOWED
dry_run_plan


## Outputs / receipt schema / claim boundary

未来 receipt：P2 upstream hash、projector source/config/parameter counts、frozen per-lane eligibility matrix、真实 patient-ID provenance，以及每 lane 的 `eligible_rows`、`bypass_rows`、`unique_patients`、`projector_calls`、`native_only_calls`、trainable parameter counts、matched-capacity proof、budget/seed/selection、native predictions/metrics、label-free outer lineage、verification 与 decision。Verifier 必须证明 ineligible lanes 的 projector calls=0，并把它们限定为 P2 native parity/guardrail。

**Claim boundary：** P9−P2 只能归因为真实 patient-ID eligible lanes 上 PAFA projector 的受控 capacity effect；ineligible lanes 不提供 projector-effect claim。P9 不是 PAFA loss attribution、official reproduction、zero-shot 或 full-encoder evidence。

**Test Result=Not run**  
**Decision：Not evaluated；Design / Not Ready。**